In [ ]:
import pandas as pd
import torch
from sklearn.preprocessing import QuantileTransformer, LabelEncoder
import os

# Set paths
input_csv = "../data/Car_Hacking_5%.csv"
output_file = "../data/raw_chunks.pt"

# Load CSV
df = pd.read_csv(input_csv)

# Select numeric features (CAN ID + 8 data bytes)
feature_cols = df.select_dtypes(include="number").columns.tolist()

# Normalize features using QuantileTransformer
scaler = QuantileTransformer()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

# Label encode string labels (e.g., "R", "DoS", "gear", etc.)
le = LabelEncoder()
df["Label"] = le.fit_transform(df["Label"])

# Chunk parameters
chunk_size = 27
X, y = [], []

# Create chunks per label
for label in df["Label"].unique():
    df_label = df[df["Label"] == label][feature_cols]
    chunks = [
        df_label.iloc[i : i + chunk_size].values
        for i in range(0, len(df_label) - chunk_size + 1, chunk_size)
    ]
    X.extend(chunks)
    y.extend([label] * len(chunks))

# Convert to torch tensors
X_tensor = torch.tensor(X, dtype=torch.float32)  # Shape: [N, 27, 9]
y_tensor = torch.tensor(y, dtype=torch.long)  # Shape: [N]

# Save
torch.save((X_tensor, y_tensor), output_file)

print(f"✅ Saved {X_tensor.shape[0]} samples to '{output_file}'")

✅ Saved 30311 samples to '../data/raw_chunks.pt'


C:\Users\SakifKhan\AppData\Local\Temp\ipykernel_32628\1589519615.py:44: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  X_tensor = torch.tensor(X, dtype=torch.float32)  # Shape: [N, 27, 9]
